In [ ]:
import hashlib
from collections import defaultdict

import duckdb
import pandas as pd
from unidecode import unidecode

# Lecture de ma base de données

Tout ce qui est contenu sur mon drive


In [2]:
# Convertir la liste de fichiers en DataFrame
df = pd.read_csv(
    "../data/files.csv",
    delimiter=";",
    encoding="utf-8",
)

In [3]:
df.head(3)

,mimeType,id,name,modifiedTime,parents,size
0,application/vnd.google-apps.folder,0B1AgTg4MMq5DQTBvVXNaY1RrR00,Annales 2A ENSEM,2025-04-25T14:17:06.891Z,NaN,NaN
1,application/pdf,1-3_iXdKTJJobl2x_t3UPyUaRQeK6297b,CV Guillaume Poirier - Ingénieur - Chargé de P...,2025-04-14T13:05:47.000Z,['1AsxFlrdeAHUpOq43dZI5huH-HBHTuVov'],238845.0
2,application/vnd.google-apps.spreadsheet,10uSo7MqXrrmt7bld--NwUQsaMiJKPaz_D2EXFk9Xxvw,Organisation rando 2025,2025-04-10T09:27:43.751Z,['1zN3Ud-aqh7i_LQrnzOddJCsCkoz8Fa1V'],125305.0


In [ ]:
# Correction du pré-traitement du parent_id
df["parent_id"] = df["parents"].apply(lambda x: x[2:-2] if isinstance(x, str) else None)

In [5]:
# 2. Organisation des données
elements = df.set_index("id").to_dict(orient="index")

In [6]:
# 3. Construction de l'arbre parent → enfants
arbre = defaultdict(list)
for element_id, data in elements.items():
    parent_id = data["parent_id"]
    if parent_id:
        arbre[parent_id].append(element_id)

In [7]:
# 4. Fonctions de hash
def hash_fichier(file_data):
    """Hash pour un fichier basé sur nom + taille."""
    name = file_data["name"]
    size = str(file_data["size"]) if not pd.isna(file_data["size"]) else "0"
    base = f"{name}|{size}"
    return hashlib.sha1(base.encode()).hexdigest()

In [8]:
def hash_dossier(dossier_id):
    """Hash récursif pour un dossier."""
    enfants = arbre.get(dossier_id, [])
    hashes = []
    for enfant_id in enfants:
        enfant = elements[enfant_id]
        mimeType = enfant["mimeType"]
        if mimeType == "application/vnd.google-apps.folder":
            h = hash_dossier(enfant_id)
        else:
            h = hash_fichier(enfant)
        hashes.append((enfant["name"], h))

    # Tri pour que l'ordre n'importe pas
    hashes.sort()
    concat = "".join(f"{nom}:{h}" for nom, h in hashes)
    return hashlib.sha1(concat.encode()).hexdigest()

In [9]:
# 5. Calcul du hash pour chaque dossier racine
dossier_hash = {}

for element_id, data in elements.items():
    if data["mimeType"] == "application/vnd.google-apps.folder":
        dossier_hash[element_id] = hash_dossier(element_id)

In [10]:
# 6. Groupement par hash
doublons = defaultdict(list)
for dossier_id, h in dossier_hash.items():
    doublons[h].append(dossier_id)

In [ ]:
# 7. Affichage des doublons (noms + ids)
for h, ids in doublons.items():
    if len(ids) > 1:
        noms_ids = [f"{elements[id]['name']} (id={id})" for id in ids]
        print(f"Dossiers doublons trouvés : {noms_ids}")

2
Dossiers doublons trouvés : ['TP Compression _ Transmission Information (id=18ByQ9ltq0KI3HeQanu58vcB7q7nLpr2k)', 'TP Compression _ Transmission Information (id=1rWw9fYB3Yz5l_iVlHoj2ra9Wv_7uLbkh)']
2
Dossiers doublons trouvés : ['TP compression (id=1E3TYHjhCv56b48ovs0aakm6Y9sIAMrOW)', 'TP compression (id=1E4ygfKYEJtqOYBb9EUZkbpE1YKbwy8Lc)']
2
Dossiers doublons trouvés : ['Matlab (id=1A-iMyjqCJ74A3FaUowk9rnNskPzhaveF)', 'Matlab (id=11c75qmU0HiAmJ-n3YyCnYeIDUTqYGDW2)']
2
Dossiers doublons trouvés : ['TD (id=1p3h7tTagljMZrKQbve7ffxLNixm9ImFH)', 'TD (id=1AP-EZAbeRVrXM116jvDPAdewgA8UhcAk)']
3
Dossiers doublons trouvés : ['Annales (id=1xEbjJaBp6aDl3nuJ4iafBOr7l7V7QWH6)', 'Annales (id=1b9ngF-MWPiWMEVFbWxqRDS38lcrcP-Hn)', 'Annales (id=1zDy-PC2qRMRyQfBjnlqV4i0gPYlAb9VT)']
2
Dossiers doublons trouvés : ['scenarios (id=1aYx0wqfw2vTjtr8vRGDc90KbMXLVnZ3O)', 'scenarios (id=1y92YuSoTqeIgyXLEs5HuqsqhmWurRSWB)']
2
Dossiers doublons trouvés : ['Thursday_June_01_091939 (id=1yxExkmoGJ9374wW8USkfpqIH2gkcB

In [13]:
doublons

defaultdict(list,
            {'ee081dddf871311f8d04145761cb175378f2f5b9': ['0B1AgTg4MMq5DQTBvVXNaY1RrR00'],
             '5e26cde41f27b76bcbe411e13bee5a68495392ba': ['1zN3Ud-aqh7i_LQrnzOddJCsCkoz8Fa1V'],
             '92fdb12f032cf04beb385aaff3b37a2d6bbad7c0': ['18ByQ9ltq0KI3HeQanu58vcB7q7nLpr2k',
              '1rWw9fYB3Yz5l_iVlHoj2ra9Wv_7uLbkh'],
             'cddfbff9f21c8c66976d7b5b564649c99e5c991b': ['1E3TYHjhCv56b48ovs0aakm6Y9sIAMrOW',
              '1E4ygfKYEJtqOYBb9EUZkbpE1YKbwy8Lc'],
             '11cf78f5297b481d429da40d21c8b978a571b721': ['1jGUuHNq-FU2GIYZaqJG9VyVs9GzOUUP5'],
             '85b05c9968627bab89cf47d3f841e7f3dbb4ca58': ['1A-iMyjqCJ74A3FaUowk9rnNskPzhaveF',
              '11c75qmU0HiAmJ-n3YyCnYeIDUTqYGDW2'],
             '965e3bfb330f2c053241471264e44d0e456c4567': ['1TkY6iq1sBnGOiJu6Lbc5K47KxSr0aARC'],
             'ee18f2a54139f575fc4d208e4f480908a60d6183': ['1p3h7tTagljMZrKQbve7ffxLNixm9ImFH',
              '1AP-EZAbeRVrXM116jvDPAdewgA8UhcAk'],
             'd

# Test de dedoublonnage sur les dossiers

Filtrage pour n'avoir que les dossiers et sans les éléments partagés


In [ ]:
query = """
SELECT
  SUBSTR(parents, 3, LENGTH(parents) - 4) AS parent_id,
  LIST(name) AS names_list,
  LIST(id) AS ids_list,
  SUM(
    CASE WHEN size THEN size
    ELSE 0
    END
  ) AS total_size,
  COUNT(id) AS count_files,
FROM df
WHERE
  parent_id IS NOT NULL
  -- AND mimeType == 'application/vnd.google-apps.folder'
GROUP BY parent_id
ORDER BY total_size DESC
"""

df_test_1 = duckdb.sql(query).df()

In [ ]:
df_test_1.drop_duplicates(subset=["names_list"], keep="first", inplace=True)

In [ ]:
query = """WITH folder_contents AS (
  SELECT
  SUBSTR(parents, 3, LENGTH(parents) - 4) AS parent_id,
  LIST(name) AS names_list,
  LIST(id) AS ids_list,
  SUM(
    CASE
      WHEN size THEN size
    ELSE 0
    END
  ) AS total_size,
  COUNT(id) AS count_files,
FROM df
WHERE
  parent_id IS NOT NULL
  -- AND mimeType == 'application/vnd.google-apps.folder'
GROUP BY parent_id
)

SELECT
  a.parent_id AS parent_id,
  b.parent_id AS parent_id_to_compare,
  c.name as parent_name_of_a,
  a.names_list AS names_list,
  b.names_list AS name_to_compare,
  a.ids_list AS ids_list,
  b.ids_list AS ids_list_to_compare,
  a.total_size AS total_size,
  b.total_size AS total_size_to_compare,
FROM
  folder_contents a
INNER JOIN
  folder_contents b
ON
  a.names_list = b.names_list
  and a.parent_id != b.parent_id
LEFT JOIN df c
ON
  a.parent_id = c.id
ORDER BY total_size DESC
"""
df_test_2 = duckdb.sql(query).df()

In [ ]:
df_test_2.head(15)

In [ ]:
query = """WITH folder_contents AS (
  SELECT
  SUBSTR(parents, 3, LENGTH(parents) - 4) AS parent_id,
  LIST(name) AS names_list,
  LIST(id) AS ids_list,
  SUM(
    CASE
      WHEN size THEN size
      ELSE 0
    END
  ) AS total_size,
  COUNT(id) AS count_files,
FROM df
WHERE
  parent_id IS NOT NULL
  -- AND mimeType == 'application/vnd.google-apps.folder'
GROUP BY parent_id
),

tree AS (
SELECT
  a.name AS name,
  a.id as id,
  b.name AS parent_name,
  SUBSTR(a.parents, 3, LENGTH(a.parents) - 4) AS parent_id,
  SUBSTR(b.parents, 3, LENGTH(b.parents) - 4) AS grand_parent_id,
FROM df a
INNER JOIN df b
ON SUBSTR(a.parents, 3, LENGTH(a.parents) - 4) = b.id
WHERE
  a.parents IS NOT NULL
  AND b.parents IS NOT NULL
)

SELECT
  t.parent_id AS parent_id,
  t.grand_parent_id AS grand_parent_id,
  t.name AS name,
  t.parent_name AS parent_name,
  f.names_list AS names_list,
  f.ids_list AS ids_list,
  f.total_size AS total_size,
  f.count_files AS count_files,
FROM tree t
INNER JOIN folder_contents f
  ON f.parent_id = t.parent_id
WHERE
  t.parent_id IS NOT NULL
  AND t.grand_parent_id IS NOT NULL
ORDER BY f.total_size DESC
"""
df_test_3 = duckdb.sql(query).df()

#   SUBSTR(d.parents, 3, LENGTH(d.parents) - 4) AS parent_id,
#   d.name,
#   f.names_list AS names_list,
#   -- f.names_list || LIST(d.name) AS names_list_concat,
#   f.ids_list,
#   f.total_size,
#   f.count_files
# FROM df d
# INNER JOIN folder_contents f
#   ON f.parent_id = d.id
# WHERE
#   d.mimeType = 'application/vnd.google-apps.folder'
#   AND d.parents IS NOT NULL
# ORDER BY f.total_size DESC

In [ ]:
df_test_3.head(10)

In [ ]:
df_test_3.loc[0, "names_list"]

In [ ]:
query = """WITH RECURSIVE folder_tree AS (
  SELECT
    -- Extraire l'ID du parent à partir de la chaîne de caractères
    SUBSTR(parents, 3, LENGTH(parents) - 4) AS parent_id,
    
    -- Identifiant du fichier
    id,
    
    -- Nom du fichier
    name,
    
    -- Concatenation des noms de fichiers / dossiers contenu par le parent
    ARRAY_AGG(name) OVER(
      PARTITION BY parents
    ) AS names_array,
    
    -- Concaténation des IDs de fichiers / dossiers contenu par le parent
    ARRAY_AGG(id) OVER(
      PARTITION BY parents
    ) AS ids_array,
    
    -- Somme de la taille des fichiers contenu par le parent
    SUM(COALESCE(size, 0)) OVER(
      PARTITION BY parents
    ) AS total_size,
    
    -- Compte le nombre de fichiers contenu par le parent
    COUNT(id) OVER(
      PARTITION BY parents
    ) AS count_files_folders
  FROM df
  WHERE parent_id IS NOT NULL

  UNION ALL

  SELECT
    -- ID du parent
    children_ft.parent_id,
    
    -- Identifiant du fichier / dossier
    children_ft.id,

    -- Nom du fichier / dossier
    children_ft.name,

    -- Concaténation des noms de fichiers / dossiers contenu par le parent
    parent_ft.names_array || children_ft.names_array AS names_array,

    -- Concaténation des IDs de fichiers / dossiers contenu par le parent
    parent_ft.ids_array || children_ft.ids_array AS ids_array,

    -- Somme de la taille des fichiers contenu par le parent
    parent_ft.total_size + COALESCE(children_ft.total_size, 0) AS total_size,

    -- Compte le nombre de fichiers contenu par le parent
    parent_ft.count_files_folders + children_ft.count_files_folders AS count_files_folders,
    
  -- Récursivement, on continue à explorer les parents
  FROM
    folder_tree parent_ft
  INNER JOIN
    folder_tree children_ft
  ON
    parent_ft.id = children_ft.parent_id
)

-- Sélection finale : trouver les dossiers ayant les mêmes éléments enfants mais des parents différents
SELECT
  a.parent_id AS parent_folder1,
  b.parent_id AS parent_folder2,
  a.name AS folder_name,
  a.total_size,
  a.count_files_folders,
  a.names_array
FROM folder_tree a
JOIN folder_tree b
  ON a.names_array = b.names_array
  AND a.parent_id != b.parent_id
ORDER BY a.total_size DESC;
"""
df_test_4 = duckdb.sql(query).df()

In [ ]:
df_test_4

In [ ]:
df_test_4[df_test_4["name"].isin(["2016", "2019", "2018", "2017", "2015", "2014"])]

In [ ]:
fichiers = [
    {"id": "1", "parent_id": None, "nom": "A", "type": "folder", "taille": 0},
    {"id": "2", "parent_id": "1", "nom": "fichier1.txt", "type": "file", "taille": 100},
    {"id": "3", "parent_id": "1", "nom": "fichier2.txt", "type": "file", "taille": 200},
    {"id": "4", "parent_id": None, "nom": "B", "type": "folder", "taille": 0},
    {"id": "5", "parent_id": "4", "nom": "fichier1.txt", "type": "file", "taille": 100},
    {"id": "6", "parent_id": "4", "nom": "fichier2.txt", "type": "file", "taille": 200},
    {"id": "7", "parent_id": None, "nom": "C", "type": "folder", "taille": 0},
    {"id": "8", "parent_id": "7", "nom": "fichier3.txt", "type": "file", "taille": 300},
]

In [ ]:
from collections import defaultdict

# Organisation par id
elements = {f["id"]: f for f in fichiers}

# Construction de l'arbre
arbre = defaultdict(list)
for f in fichiers:
    if f["parent_id"] is not None:
        arbre[f["parent_id"]].append(f["id"])

In [ ]:
def hash_fichier(fichier):
    # Pour un fichier simple : basé sur nom + taille
    base = f"{fichier['nom']}|{fichier['taille']}"
    return hashlib.sha1(base.encode()).hexdigest()


def hash_dossier(dossier_id):
    enfants = arbre.get(dossier_id, [])
    hashes = []
    for enfant_id in enfants:
        enfant = elements[enfant_id]
        if enfant["type"] == "file":
            h = hash_fichier(enfant)
        elif enfant["type"] == "folder":
            h = hash_dossier(enfant_id)
        else:
            continue  # ignore type inconnu
        hashes.append((enfant["nom"], h))

    # Trie par nom pour que l'ordre n'importe pas
    hashes.sort()
    concat = "".join(f"{nom}:{h}" for nom, h in hashes)
    return hashlib.sha1(concat.encode()).hexdigest()


In [ ]:
dossier_hash = {}

# On parcourt tous les dossiers racines (pas de parent_id)
for f in fichiers:
    if f["type"] == "folder" and f["parent_id"] is None:
        dossier_hash[f["id"]] = hash_dossier(f["id"])

In [ ]:
from collections import defaultdict

# Groupement par hash
doublons = defaultdict(list)
for dossier_id, h in dossier_hash.items():
    doublons[h].append(dossier_id)

# Affichage
for h, ids in doublons.items():
    if len(ids) > 1:
        print(f"Dossiers doublons : {ids}")


In [ ]:
df_test_4[df_test_4["parent_id"] == "1at09_zMtj7M8j3-pDkoRoWFDDAQt-O47"]

In [ ]:
df_test_4[df_test_4["id"] == "1at09_zMtj7M8j3-pDkoRoWFDDAQt-O47"]

In [ ]:
display(df_test_4[df_test_4["id"] == "1-9bQdI7dsJruWw4oV4x8tmmiQTeDbPDL"])
display(df_test_4[df_test_4["id"] == "1Q0cT7mQeHOCvHoistZj2yRZRFbwPPia9"])

In [ ]:
df_test_4.loc[7604, "names_array"]

In [ ]:
query = """WITH RECURSIVE folder_contents AS (
  SELECT
    SUBSTR(parents, 3, LENGTH(parents) - 4) AS parent_id,
    ARRAY_AGG(name) AS names_array,
    ARRAY_AGG(id) AS ids_array,
    SUM(COALESCE(size, 0)) AS total_size,
    COUNT(id) AS count_files_folders
  FROM df
  WHERE parents IS NOT NULL
  GROUP BY parent_id

  UNION ALL

  SELECT
    parent_ft.parent_id,
    ARRAY(SELECT DISTINCT name FROM UNNEST(parent_ft.names_array || ARRAY[d.name])) AS names_array,
    ARRAY(SELECT DISTINCT id FROM UNNEST(parent_ft.ids_array || ARRAY[d.id])) AS ids_array,
    parent_ft.total_size + COALESCE(d.size, 0) AS total_size,
    parent_ft.count_files_folders + 1 AS count_files_folders
  FROM folder_contents parent_ft
  INNER JOIN df d
  ON parent_ft.parent_id = SUBSTR(d.parents, 3, LENGTH(d.parents) - 4)
)

SELECT * FROM folder_contents
ORDER BY total_size DESC;

"""

df_test_5 = duckdb.sql(query).df()

In [ ]:
df_test_5

In [ ]:
df[df["id"] == "1vWwmK4eLzX_yb2yxz3IVtMqfCqZ9Lw9l"]

# Filtrage pour enlever les dossiers

C'est tout à fait normal que les dossiers soit en doublons si des arborescences se ressemble


In [ ]:
df_folds = df[df["mimeType"] == "application/vnd.google-apps.folder"]

In [ ]:
df_folds.groupby("name")["name"].count().reset_index(name="count").query(
    "count < 6 & count > 1"
)

In [ ]:
df = df[df["mimeType"] != "application/vnd.google-apps.folder"]

# Est-ce qu'il y a des doublous ?

Test simple sur le noms des fichiers


In [ ]:
# Regex constants
DUPLICATE_PATTERN = r"\(\d{1,2}\)|\bcopie\s*de\b"
EXTENSION_CLEANUP_PATTERN = r"\s+((\.\w+)+)$"
EXTENSION_CLEANUP_REPLACEMENT = r"\1"

In [ ]:
# Normalisation du nom
df["normalized_name"] = df["name"].fillna("").apply(unidecode)

# Création du base_name pour tout le monde
df["base_name"] = (
    df["normalized_name"]
    .str.replace(DUPLICATE_PATTERN, "", regex=True)
    .str.replace(EXTENSION_CLEANUP_PATTERN, EXTENSION_CLEANUP_REPLACEMENT, regex=True)
    .str.strip()
    .str.lower()
)

In [ ]:
def mark_duplicates(folder):
    # Ceux que l’on garde dans ce dossier
    keep = folder.sort_values(by="modifiedTime").drop_duplicates(
        subset="base_name", keep="last"
    )
    # Ceuxw que l’on supprime dans ce dossier
    folder["to_remove"] = ~folder.index.isin(keep.index)
    return folder

In [ ]:
# df = df.groupby('parents', group_keys=False).apply(mark_duplicates)

In [ ]:
df[df["base_name"] == "faceavant.png"]
df[df["base_name"] == "TPNRJ1A etn 4 Cld_31(1).pdf"]